In [ ]:
import torch
from PIL import Image, ImageOps
from transformers import DetrImageProcessor, DetrForObjectDetection
import cv2 
import csv 
from ultralytics import YOLO 
processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50", revision="no_timm")
model = DetrForObjectDetection.from_pretrained("isalia99/detr-resnet-50-sku110k")
model.eval()

def image_detection(frame):
    image = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    image = ImageOps.exif_transpose(image)
    image = image.convert("RGB")
    
    
    inputs = processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    
    target_sizes = torch.tensor([image.size[::-1]])
    results = processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=0.7)[0]
    
    bbox = []
    for box in results["boxes"]:
        box_np = box.cpu().numpy().astype(int)  
        bbox.append(box_np)
    
    return bbox 


with open('data.csv', 'w') as new_file:
    field = ['bbox1', 'bbox2', 'bbox3', 'bbox4', 'object_id']
    csv_writer = csv.DictWriter(new_file, fieldnames=field)
    csv_writer.writeheader()


model = YOLO("yolov8n.pt")
counter = 1 
webcam = cv2.VideoCapture(0) 
dataset_captured = False

with open('data.csv', 'a') as file: 
    csv_writer = csv.writer(file)
    while True: 
     ret, frame = webcam.read() 
    
     if not ret:
       break
    
     if not dataset_captured:
      results = model(frame)
      human_detected = False
        
      for box in results[0].boxes: 
            if int(box.cls[0].item()) == 0:
                human_detected = True
                break
        
     if not human_detected:
         bbox = image_detection(frame)
         with open('data.csv', 'a') as file: 
                csv_writer = csv.writer(file)
                for box in bbox: 
                 csv_writer.writerow([box[0], box[1], box[2], box[3], counter]) 
                 counter += 1 
            
         dataset_captured = True

    
     cv2.imshow('Shelf Monitor', frame)
     if dataset_captured:
        break
     if cv2.waitKey(1) & 0xFF == ord('q'):
        break

webcam.release()
cv2.destroyAllWindows()


In [ ]:
from ultralytics import YOLOWorld
lifting_para = YOLOWorld('yolov8s-world.pt') 
lifting_para.set_classes([ "shopping cart" ,"pocket on clothing " , "Billing Counter "])

In [ ]:
from ultralytics import YOLO
import cv2
import mediapipe as mp 
from deep_sort_realtime.deepsort_tracker import DeepSort
import pandas as pd 
import numpy as np 
import datetime
import face_recognition
import csv
with open('object_human_id_info.csv', 'w') as new_file:
    field = ['datetime' , 'human id' , 'object_id' ]
    csv_writer = csv.DictWriter(new_file, fieldnames=field)
    csv_writer.writeheader()

with open('Face_data_set.csv', 'w') as new_file:
    field = [ 'face encodings' , 'human id ' ]
    csv_writer = csv.DictWriter(new_file, fieldnames=field)
    csv_writer.writeheader()

mp_pose_det = mp.solutions.pose.Pose()
person_id = 0 
human_id = {}
def human_face_dataset(frame, hand_bbox):
    height, width = frame.shape[:2]
    x1, y1, x2, y2 = hand_bbox
    diff_width, diff_height = x2 - x1, y2 - y1
    target_x1 = max(0, int(x1 - diff_width*2))
    target_y1 = max(0, int(y1 - diff_height*4))
    target_x2 = min(width, int(x2 + diff_width*2))
    target_y2 = min(height, int(y2))
    target_img = frame[target_y1:target_y2, target_x1:target_x2]
    rgb = cv2.cvtColor(target_img, cv2.COLOR_BGR2RGB)
    pose = mp_pose_det.process(rgb)
    detection = pose.pose_landmarks.landmark
    nose = detection[0]
    target_h, target_w = target_img.shape[:2]
    face_bbox_x1 = int((nose.x - 0.1) * target_w)
    face_bbox_y1 = int((nose.y - 0.15) * target_h)
    face_bbox_x2 = int((nose.x + 0.1) * target_w)
    face_bbox_y2 = int((nose.y + 0.15) * target_h)
    face_est_bbox = target_img[face_bbox_y1:face_bbox_y2, face_bbox_x1:face_bbox_x2]
    if face_est_bbox.size == 0:
        return
    face_rgb = cv2.cvtColor(face_est_bbox, cv2.COLOR_BGR2RGB)
    
    face_locations = face_recognition.face_locations(face_rgb)
    face_encodings = face_recognition.face_encodings(face_rgb, face_locations)
    
    if len(face_encodings) == 0:
        return
    
    face_encoding = face_encodings[0]
    human_id[object_id] = person_id
    with open('Face_data_set.csv', 'a') as f:
        writer = csv.writer(f)
        writer.writerow([face_encoding.tolist() , person_id])
        person_id = person_id + 1 

model = YOLO('yolo-100doh.pt')
webcam = cv2.VideoCapture(0)
df = pd.read_csv('data.csv')
while True:
    ret, frame = webcam.read()
    if not ret:
        break
    
    results = model(frame, conf=0.7)
    hands_bbox = []      
    objects_bbox = []    
    objects_id = []
    Happen_status = False 
    
    for box in results[0].boxes:
        object_id = int(box.cls[0])
        bbox = box.xyxy[0].tolist()
        if object_id == 0:  
            hands_bbox.append(bbox)
        else:
            mask = np.isclose(df[['bbox1', 'bbox2', 'bbox3', 'bbox4']], bbox, atol=1.0).all(axis=1)
            matched_rows = df.loc[mask, 'object_id']
            o_id = matched_rows.values[0]
            objects_bbox.append(bbox)
            objects_id.append(o_id)
        with open('id_detection.csv', 'a') as f:
           writer = csv.writer(f)
           writer.writerow([datetime.now(), hands_bbox, objects_bbox, objects_id])

    if  len(hands_bbox) > 0 and len(objects_bbox) > 0 : 
        Happen_status = True 
    if Happen_status is True : 
        for hand in hands_bbox : 
           human_face_dataset(frame , hands_bbox)
        Tracker   = DeepSort(objects_bbox , frame = frame )
        monitoring_objects(objects_id)
    with open('object_human_id_info.csv', 'a') as f:
           writer = csv.writer(f)
           for object in object_id : 
               writer.writerow([datetime.now(), human_id[object], object])

    cv2.imshow('Detection', results[0].plot())
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

webcam.release()
cv2.destroyAllWindows()


In [ ]:
with open('suspcious.csv', 'w') as new_file:
    field = ['datetime' , 'human id' , 'object_id' ]
    csv_writer = csv.DictWriter(new_file, fieldnames=field)
    csv_writer.writeheader()

safe_id = {}
suspicious_id = {}  
tracking_time = {}
def inside_cart(cart_bbox , bbox  , object_id  ) : 
    for bbox in cart_bbox : 
       x1 , y1 , x2 , y2 = cart_bbox
       a  = bbox[0] 
       b = bbox[1]
       if a >=  x1 and a <= x2 and b <= y2 and b >= y1 : 
        safe_id[object_id] = 1
        suspicious_id[object_id] = 0 

def inside_pocket(pocket_bbox , bbox , object_id  ) : 
    for bbox in pocket_bbox : 
       x1 , y1 , x2 , y2 = pocket_bbox
       a  = bbox[0] 
       b = bbox[1]
       if a >=  x1 and a <= x2 and b <= y2 and b >= y1 : 
        suspicious_id[object_id] = suspicious_id[object_id]+  0.5 

def velocity_of_objects(bbox ,prev_positions,track.track_id ) : 
    x = bbox[0] - prev_positions[track.track_id][0]
    y = bbox[1] - prev_positions[track.track_id][1]
    speed = (x**2 + y**2)**(1/2)
    if (speed > 50 and safe_id[track.track_id] != 1 )  : 
      suspicious_id[track.track_id] = suspicious_id[track.track_id] + 0.25 
    prev_positions[track.track_id][0] = bbox[0] 
    prev_positions[track.track_id][1] = bbox[1]
      
   

prev_positions = {}
def monitoring_objects(object_id ) : 
    i= 0 
    tracks  = Tracker.update_tracks() 
    for track in tracks : 
        track.track_id  = object_id[i]
        i = i+1 
    while  len(tracks) > 0  : 
        ret , frame = webcam.read()
        if (len(tracks) > 0 ): 
           for track in tracks : 
              prev_positions[track.track_id].append([track[0] , track[1]])
           
        tracks  = Tracker.update_tracks() 
        if len(tracks) > 0 : 
          results = model.predict(frame)
          for track in tracks :
              x1,y1,x2,y2 = track 
              center_x = (x1+x2)/2
              center_y = (y1+y2)/2 
              track_bbox  = []
              track_bbox.append([center_x , center_y])
              if (safe_id[track.track_id] != 1 and tracking_time[track.track_id] < 5400 ) : 
                 tracking_time[track.track_id]  = tracking_time[track.track_id] + 1 
                 inside_cart(track_bbox , results[0] , track.track_id)
              
              if (suspicious_id[track.track_id] <= 0.75 and  tracking_time[track.track_id] < 5400 ): 
                 tracking_time[track.track_id]  = tracking_time[track.track_id] + 1 
                 inside_pocket(track_bbox , results[1] , track.track_id )
              
              if( suspicious_id[track.track_id] <= 0.75 and tracking_time[track.track_id] < 5400 and len(prev_positions) >= 0   ) :
                  tracking_time[track.track_id]  = tracking_time[track.track_id] + 1 
                  velocity_of_objects(track_bbox ,prev_positions ,  track.track_id)
              if (suspicious_id[track.track_id] > 0.75 and safe_id[track.track_id] != 1 ) : 
                      with open('suspcious.csv', 'a') as f:
                       writer = csv.writer(f)
                       writer.writerow([datetime.now(), human_id[track.track_id], track.track_id])

In [ ]:
import cv2 
import pandas as pd 
from deep_sort_realtime.deepsort_tracker import DeepSort
import numpy as np 

df = pd.read_csv('Face_data_set.csv')
sus_data =  pd.read_csv('suspcious.csv')
sus_human_id = sus_data[1]
results = model.predict(frame)
sus_human_encodings  = {}
for id in sus_human_id : 
      row = df[df["id"] == id].iloc[0]
      encoding = np.array(eval(row["face encodings"]))  
      sus_human_encodings.append(encoding)
webcam = cv2.videocapture(0) 
Tracker   = DeepSort(sus_human_encodings , frame = frame )
tracks  = Tracker.update_tracks() 
i = 0 
for track in tracks : 
     track.tracks_id = sus_human_id[i]
     i = i+ 1  


billing_zones = {}
def zones(counter_bbox) : 
      bil_zone = []
      for zone in counter_bbox : 
          x1,y1,x2,y2 = zone 
          x1 = x1+ 100 
          y1 = y1+ 100 
          x2 = x2+ 100 
          y2 = y2+ 100 
          bil_zone.append([x1,y1,x2,y2])
      billing_zones.append(bil_zone)
      return billing_zones

time_in_zone = {}
def cal_time(track , safe_zone) : 
      for bbox in safe_zone : 
       x1 , y1 , x2 , y2 = safe_zone
       a  = track[0] 
       b = track[1]
       if a >=  x1 and a <= x2 and b <= y2 and b >= y1 : 
         time_in_zone[track.tracks_id] =  time_in_zone[track.tracks_id] + 1 
         return 1 
       return 0 
safe_people_id = []
hash = {}
while ( len(sus_human_id) > 0 ) : 
    ret , frame = webcam.read() 
    tracks  = Tracker.update_tracks() 
    counter_bbox = results[2]
    safe_zone  = zones(counter_bbox)
    for track in tracks : 
        if (hash[track.tracks_id] != 1)  : 
         if (cal_time(track , safe_zone) == 0 ) : 
            if (time_in_zone[track.tracks_id] > 2700 ) : 
                safe_people_id.append(track.tracks_id)
                hash[track.tracks_id] = hash[track.tracks_id] +1 
        
    email_id = sus_human_id - safe_people_id 

         


In [ ]:
import smtplib, ssl
import pandas as pd 
def email () :
     df = pd.read_csv('person_info.csv')
     email_id  =  df[1] 
     sender_email = "varadbendale17@gmail.com"
     receiver_list = email_id
     subject = "Official Notice: Store Security Incident"
     body = "Hello, records show you are suspected for shoplifting during your recent visit. Please contact us to resolve this."
     message = f"Subject: {subject}\n\n{body}"
     context = ssl.create_default_context()
     with smtplib.SMTP_SSL("smtp.gmail.com", 465, context=context) as server:
      server.login(sender_email, password)
      for receiver in receiver_list:
        server.sendmail(sender_email, receiver, message)
